# 03 Process Domestic EPC Data for demand model

## 1. Import libraries

In [ ]:
#import the necessary libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
PROJECT_DIR = next(candidate
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (candidate / "01_Data").exists()
                   )

In [ ]:
RAW_EPC_DIR = PROJECT_DIR / "01_Data/Raw/EPC_Domestic_Plymouth"
ONS_LOOKUP_DIR = PROJECT_DIR / "01_Data/Raw/ONS_Lookups"

PROCESSED_EPC_DIR = PROJECT_DIR / "01_Data/Processed/EPC"
PROCESSED_ENERGY_DIR = PROJECT_DIR / "01_Data/Processed/Energy"

PROCESSED_EPC_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_ENERGY_DIR.mkdir(parents=True, exist_ok=True)

## 2. Define input file paths


In [ ]:
EPC_CERTIFICATES_PATH = RAW_EPC_DIR / "certificates.csv"

POSTCODE_LOOKUP_PATH = (ONS_LOOKUP_DIR
    / "PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv"
)

ENERGY_LSOA_PATH = ( PROCESSED_ENERGY_DIR
    / "plymouth_lsoa_energy_consumption_2024.csv"
)

RAW_FUEL_PRICE_DIR = PROJECT_DIR / "01_Data/Raw/BRE_SAP_fuel_prices"

FUEL_PRICE_HISTORY_PATH = RAW_FUEL_PRICE_DIR / "bre_sap_fuel_prices_relevant_2012_2026_wide.csv"


## 3. Load the domestic EPC dataset


In [ ]:
#required columns 
epc_usecols = [
    "LMK_KEY",
    "BUILDING_REFERENCE_NUMBER",
    "UPRN",
    "UPRN_SOURCE",
    "POSTCODE",
    "LOCAL_AUTHORITY",
    "LOCAL_AUTHORITY_LABEL",
    "LODGEMENT_DATE",
    "INSPECTION_DATE",
    "REPORT_TYPE",
    "TRANSACTION_TYPE",
    "PROPERTY_TYPE",
    "BUILT_FORM",
    "CONSTRUCTION_AGE_BAND",
    "TENURE",
    "TOTAL_FLOOR_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "CURRENT_ENERGY_RATING",
    "POTENTIAL_ENERGY_RATING",
    "CURRENT_ENERGY_EFFICIENCY",
    "POTENTIAL_ENERGY_EFFICIENCY",
    "ENERGY_CONSUMPTION_CURRENT",
    "ENERGY_CONSUMPTION_POTENTIAL",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
    "MAINS_GAS_FLAG",
    "MAIN_FUEL",
    "MAINHEAT_DESCRIPTION",
    "SECONDHEAT_DESCRIPTION",
    "HOTWATER_DESCRIPTION",
    "MAINHEAT_ENERGY_EFF",
    "HOT_WATER_ENERGY_EFF",
    "MAIN_HEATING_CONTROLS",
]

In [ ]:
epc_raw = pd.read_csv(
    EPC_CERTIFICATES_PATH,
    usecols=epc_usecols,
    low_memory=False,
)

In [ ]:
epc_raw.head()

In [ ]:
epc_raw.shape

In [ ]:
epc_raw.isnull().sum().sort_values(ascending=False)

In [ ]:
#convert date columns 
epc_raw["LODGEMENT_DATE"] = pd.to_datetime(epc_raw["LODGEMENT_DATE"],errors="coerce")

epc_raw["INSPECTION_DATE"] = pd.to_datetime(epc_raw["INSPECTION_DATE"],errors="coerce")

In [ ]:
#check date
epc_raw[["LODGEMENT_DATE", "INSPECTION_DATE"]].head()

In [ ]:
# Keep only EPCs available by the end of the 2024 baseline year
epc_raw = epc_raw[
    epc_raw["LODGEMENT_DATE"] <= pd.Timestamp("2024-12-31")
].copy()

## 4. Create a property identifier


In [ ]:
#UPRN is used where available. Building reference number is used as thefallback identifier.
#check how many EPCs have missing UPRN values
epc_raw["UPRN"].isnull().sum()

In [ ]:
#check how many EPCs have missing building reference number values
epc_raw["BUILDING_REFERENCE_NUMBER"].isnull().sum()

In [ ]:
#clean uprn and building reference number
uprn_clean = pd.to_numeric(epc_raw["UPRN"],errors="coerce").astype("Int64").astype("string")

brn_clean = epc_raw["BUILDING_REFERENCE_NUMBER"].astype("string").str.strip()

In [ ]:
#create property identifier using building reference number
epc_raw["PROPERTY_ID"] = "BRN_" + brn_clean

In [ ]:
#use uprn where it is available
has_uprn = uprn_clean.notna()

epc_raw.loc[has_uprn,"PROPERTY_ID"] = "UPRN_" + uprn_clean[has_uprn]

In [ ]:
#check the property identifier column
epc_raw[["UPRN","BUILDING_REFERENCE_NUMBER","PROPERTY_ID",]].head()

In [ ]:
#check if property id column has null values
epc_raw["PROPERTY_ID"].isnull().sum()

## 5. Keep the latest EPC for each property


In [ ]:
#sort dataset by property id, lodgement date, and inspection date in descending order to keep the latest record for each property
epc_sorted = epc_raw.sort_values(["PROPERTY_ID","LODGEMENT_DATE","INSPECTION_DATE",],
                                 ascending=[True, False, False],
                                 kind="mergesort")

In [ ]:
epc_sorted.head()

In [ ]:
epc_sorted[epc_sorted.duplicated(subset=["PROPERTY_ID"],keep=False)].head(10)

In [ ]:
epc_latest = epc_sorted.drop_duplicates(subset="PROPERTY_ID",keep="first",).reset_index(drop=True)

In [ ]:
#check how many duplicates were removed 
print("Original EPC certificates:", len(epc_raw))
print("Latest property records:", len(epc_latest))
print("Older certificates removed:", len(epc_raw) - len(epc_latest))

In [ ]:
epc_latest.head()

In [ ]:
#check missing values after removing duplicates
missing_after_dedup =epc_latest.isnull().sum().rename("missing_values_count").to_frame()
missing_after_dedup["missing_pct"] = (missing_after_dedup["missing_values_count"]/ len(epc_latest)* 100).round(2)
missing_after_dedup = missing_after_dedup.sort_values("missing_values_count", ascending=False)
missing_after_dedup

## 6. Match EPC data to LSOA codes


In [ ]:
#load postcode data
postcode_data = pd.read_csv(POSTCODE_LOOKUP_PATH)

In [ ]:
postcode_data.columns.tolist()

In [ ]:
#identify required columns for postcode lookup
required_columns = ["pcds","lsoa21cd","lsoa21nm","ladcd","ladnm"]

In [ ]:
#filter postcode data to only include required columns
postcode_data = postcode_data[required_columns].copy()

In [ ]:
postcode_data.head()

In [ ]:
#filter postcode data to plymouth local authority only
postcode_data = postcode_data[postcode_data["ladcd"] == "E06000026"].copy()
postcode_data.shape

In [ ]:
#clean postcode column by converting to uppercase, removing spaces, and stripping whitespace
def clean_postcode(series):
    return series.astype("string").str.upper().str.replace(" ", "", regex=False).str.strip()

In [ ]:
#clean postcode column in postcode data and epc data
postcode_data["postcode_clean"] = clean_postcode(postcode_data["pcds"])
epc_latest["postcode_clean"] = clean_postcode(epc_latest["POSTCODE"])

In [ ]:
#drop duplicates from cleaned postcode data 
postcode_data = postcode_data.drop_duplicates(subset="postcode_clean", keep="first")

In [ ]:
postcode_data.columns.tolist()

In [ ]:
columns_to_keep = postcode_data.drop('pcds', axis=1).columns.tolist()

In [ ]:
#join postcode data to epc data using the cleaned postcode column
epc_merged = epc_latest.merge(postcode_data[columns_to_keep],on="postcode_clean",how="left",validate="many_to_one")

In [ ]:
#check how many EPCs were matched to LSOA codes
matched = epc_merged["lsoa21cd"].notna().sum()
unmatched = len(epc_merged) - matched

In [ ]:
print("Latest EPC properties:", len(epc_merged))
print("Matched to LSOA21:",matched,f"({matched / len(epc_merged):.2%})")
print("Unmatched:",unmatched,f"({unmatched / len(epc_merged):.2%})")

In [ ]:
#check epc records that did not match to lsoa21cd and count the number of unmatched records by postcode, displaying the top 20 postcodes with the most unmatched records
unmatched_postcodes = (epc_merged.loc[epc_merged["lsoa21cd"].isna(),"POSTCODE"].value_counts()
    .head(20).rename("unmatched_records").to_frame())
unmatched_postcodes

In [ ]:
#keep only properties with a matched LSOA code
epc_merged = epc_merged[epc_merged["lsoa21cd"].notna()].copy()

In [ ]:
#check the first few rows of the merged dataset to verify the join
epc_merged[["PROPERTY_ID","POSTCODE","lsoa21cd","lsoa21nm"]].head()

In [ ]:
#select the columns to keep for the final dataset
model_cols = [
    "PROPERTY_ID",
    "LMK_KEY",
    "BUILDING_REFERENCE_NUMBER",
    "UPRN",
    "POSTCODE",
    "postcode_clean",
    "lsoa21cd",
    "lsoa21nm",
    "ladcd",
    "ladnm",
    "LODGEMENT_DATE",
    "INSPECTION_DATE",
    "REPORT_TYPE",
    "TRANSACTION_TYPE",
    "PROPERTY_TYPE",
    "BUILT_FORM",
    "CONSTRUCTION_AGE_BAND",
    "TENURE",
    "TOTAL_FLOOR_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "CURRENT_ENERGY_RATING",
    "POTENTIAL_ENERGY_RATING",
    "CURRENT_ENERGY_EFFICIENCY",
    "POTENTIAL_ENERGY_EFFICIENCY",
    "ENERGY_CONSUMPTION_CURRENT",
    "ENERGY_CONSUMPTION_POTENTIAL",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
    "MAINS_GAS_FLAG",
    "MAIN_FUEL",
    "MAINHEAT_DESCRIPTION",
    "SECONDHEAT_DESCRIPTION",
    "HOTWATER_DESCRIPTION",
    "MAINHEAT_ENERGY_EFF",
    "HOT_WATER_ENERGY_EFF",
    "MAIN_HEATING_CONTROLS",
]

## 7. Create the modelling data


In [ ]:
epc_model = epc_merged[model_cols].copy()

In [ ]:
epc_model.head()

In [ ]:
epc_model.isnull().sum().sort_values(ascending=False)

In [ ]:
#check the data types of the columns in the epc_model dataframe
numeric_columns = [
    "TOTAL_FLOOR_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "CURRENT_ENERGY_EFFICIENCY",
    "POTENTIAL_ENERGY_EFFICIENCY",
    "ENERGY_CONSUMPTION_CURRENT",
    "ENERGY_CONSUMPTION_POTENTIAL",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
]

In [ ]:
for column in numeric_columns:
    epc_model[column] = pd.to_numeric(epc_model[column],errors="coerce")

epc_model[numeric_columns].dtypes

In [ ]:
#clean clean construction age band column by filling missing values with "Unknown"
epc_model["CONSTRUCTION_AGE_BAND"] = epc_model["CONSTRUCTION_AGE_BAND"].fillna("Unknown")

In [ ]:
epc_model["BUILT_FORM"] = epc_model["BUILT_FORM"].replace(["NO DATA!", "Not Recorded"],pd.NA).fillna("Unknown")

In [ ]:
#check floor area values to ensure they are in a good range
epc_model['TOTAL_FLOOR_AREA'].describe()

In [ ]:
#remove floor area values that are outside the acceptable range
epc_model = epc_model[epc_model['TOTAL_FLOOR_AREA'].between(20,500,inclusive="both")].copy()

In [ ]:
epc_model["TOTAL_FLOOR_AREA"].describe()

In [ ]:
#check heating cost values to ensure they are in a good range
epc_model["HEATING_COST_CURRENT"].describe()

In [ ]:
#check the lowest heating cost values 
epc_model["HEATING_COST_CURRENT"].sort_values().head(10)

In [ ]:
epc_model["HEATING_COST_CURRENT"] = epc_model["HEATING_COST_CURRENT"].replace(-1, np.nan)
epc_model["HEATING_COST_POTENTIAL"] = epc_model["HEATING_COST_POTENTIAL"].replace(-1, np.nan)
epc_model["QC_POTENTIAL_HEATING_COST"] = (epc_model["HEATING_COST_POTENTIAL"]
                                          <= 1.10 * epc_model["HEATING_COST_CURRENT"])

In [ ]:
epc_model["HOT_WATER_COST_CURRENT"] = (epc_model["HOT_WATER_COST_CURRENT"].replace(-1, np.nan))

epc_model["HOT_WATER_COST_POTENTIAL"] = (epc_model["HOT_WATER_COST_POTENTIAL"].replace(-1, np.nan))

In [ ]:
epc_model[["TOTAL_FLOOR_AREA","HEATING_COST_CURRENT","HOT_WATER_COST_CURRENT","ENERGY_CONSUMPTION_CURRENT"]].describe().T

In [ ]:
epc_model[
    [
        "PROPERTY_ID",
        "PROPERTY_TYPE",
        "TOTAL_FLOOR_AREA",
        "HEATING_COST_CURRENT",
        "MAIN_FUEL",
        "MAINHEAT_DESCRIPTION",
    ]
].sort_values("HEATING_COST_CURRENT",ascending=False).head(20)

## 8. Residential characteristics of high off-gas LSOAs


In [ ]:
# Prepare EPC and off-gas variables for the LSOA residential profile
energy_lsoa = pd.read_csv(ENERGY_LSOA_PATH)

off_gas_lsoa = (
    energy_lsoa[["LSOA_code", "LSOA", "off_gas_property_share"]]
    .rename(
        columns={
            "LSOA_code": "lsoa21cd",
            "LSOA": "lsoa21nm_desnz",
        }
    )
    .copy()
)

epc_profile = epc_model.copy()

property_type_text = (
    epc_profile["PROPERTY_TYPE"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
epc_profile["IS_FLAT_MAISONETTE"] = property_type_text.isin(
    ["flat", "maisonette"]
)

mainheat_text_profile = (
    epc_profile["MAINHEAT_DESCRIPTION"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
epc_profile["IS_COMMUNITY_HEATING"] = mainheat_text_profile.str.contains(
    "community",
    regex=False,
)

tenure_text = (
    epc_profile["TENURE"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
epc_profile["TENURE_KNOWN"] = ~tenure_text.isin(
    ["", "unknown", "no data!", "nan"]
)
epc_profile["IS_SOCIAL_RENTED"] = tenure_text.str.contains(
    "social",
    regex=False,
)

lsoa_epc_characteristics = (
    epc_profile
    .groupby(["lsoa21cd", "lsoa21nm"], as_index=False)
    .agg(
        EPC_RECORDS=("PROPERTY_ID", "size"),
        FLAT_MAISONETTE_COUNT=("IS_FLAT_MAISONETTE", "sum"),
        COMMUNITY_HEATING_COUNT=("IS_COMMUNITY_HEATING", "sum"),
        TENURE_KNOWN_COUNT=("TENURE_KNOWN", "sum"),
        SOCIAL_RENTED_COUNT=("IS_SOCIAL_RENTED", "sum"),
    )
)

lsoa_epc_characteristics["FLAT_MAISONETTE_PCT"] = (
    100
    * lsoa_epc_characteristics["FLAT_MAISONETTE_COUNT"]
    / lsoa_epc_characteristics["EPC_RECORDS"]
)
lsoa_epc_characteristics["COMMUNITY_HEATING_PCT"] = (
    100
    * lsoa_epc_characteristics["COMMUNITY_HEATING_COUNT"]
    / lsoa_epc_characteristics["EPC_RECORDS"]
)
lsoa_epc_characteristics["SOCIAL_RENTED_PCT_KNOWN_TENURE"] = (
    100
    * lsoa_epc_characteristics["SOCIAL_RENTED_COUNT"]
    / lsoa_epc_characteristics["TENURE_KNOWN_COUNT"]
)

off_gas_epc_profile = off_gas_lsoa.merge(
    lsoa_epc_characteristics,
    on="lsoa21cd",
    how="left",
    validate="one_to_one",
)

off_gas_epc_profile["OFF_GAS_PCT"] = 100 * off_gas_epc_profile["off_gas_property_share"]


In [ ]:
# Summarise residential characteristics for high off-gas LSOAs
high_off_gas_lsoas = (
    off_gas_epc_profile[off_gas_epc_profile["off_gas_property_share"] >= 0.38]
    .sort_values("off_gas_property_share", ascending=False)
    .copy()
)

high_off_gas_lsoa_table = high_off_gas_lsoas[
    [
        "lsoa21nm_desnz",
        "OFF_GAS_PCT",
        "EPC_RECORDS",
        "FLAT_MAISONETTE_PCT",
        "COMMUNITY_HEATING_COUNT",
        "COMMUNITY_HEATING_PCT",
        "TENURE_KNOWN_COUNT",
        "SOCIAL_RENTED_PCT_KNOWN_TENURE",
    ]
].copy()

print("HIGH-OFF-GAS LSOAs")
display(high_off_gas_lsoa_table.round(2))


In [ ]:
# Compare the high off-gas profile with Plymouth overall
high_off_gas_lsoa_codes = high_off_gas_lsoas["lsoa21cd"].tolist()

high_off_gas_epc = epc_profile[
    epc_profile["lsoa21cd"].isin(high_off_gas_lsoa_codes)
].copy()


def summarise_epc_profile(df, group_name):

    n = len(df)

    flat_maisonette_pct = (
        100 * df["IS_FLAT_MAISONETTE"].sum() / n
        if n > 0 else np.nan
    )

    community_heating_pct = (
        100 * df["IS_COMMUNITY_HEATING"].sum() / n
        if n > 0 else np.nan
    )

    known_tenure = df["TENURE_KNOWN"].sum()

    social_rented_pct = (
        100 * df["IS_SOCIAL_RENTED"].sum() / known_tenure
        if known_tenure > 0 else np.nan
    )

    return {
        "Group": group_name,
        "EPC_RECORDS": n,
        "FLAT_MAISONETTE_COUNT": int(df["IS_FLAT_MAISONETTE"].sum()),
        "FLAT_MAISONETTE_PCT": flat_maisonette_pct,
        "COMMUNITY_HEATING_COUNT": int(df["IS_COMMUNITY_HEATING"].sum()),
        "COMMUNITY_HEATING_PCT": community_heating_pct,
        "TENURE_KNOWN_COUNT": int(known_tenure),
        "SOCIAL_RENTED_COUNT": int(df["IS_SOCIAL_RENTED"].sum()),
        "SOCIAL_RENTED_PCT_KNOWN_TENURE": social_rented_pct,
    }


residential_profile_comparison = pd.DataFrame(
    [
        summarise_epc_profile(epc_profile, "Plymouth overall"),
        summarise_epc_profile(high_off_gas_epc, "High-off-gas LSOAs (>=38%)"),
    ]
)

print("PLYMOUTH OVERALL VS HIGH-OFF-GAS LSOAs")
display(residential_profile_comparison.round(2))


## 9. Classify the main heating system


In [ ]:
# Preprocess main fuel and main heating description text columns
main_fuel_text = epc_model["MAIN_FUEL"].fillna("").astype("string").str.strip().str.lower()

mainheat_text = epc_model["MAINHEAT_DESCRIPTION"].fillna("").astype("string").str.strip().str.lower()


combined_heating_text = main_fuel_text+ " " + mainheat_text
combined_heating_text.head()

In [ ]:
#create heating system category 
epc_model["HEATING_SYSTEM_CLASS"] = "Unknown"

In [ ]:
#classify electricity heating systems
electricity_category = combined_heating_text.str.contains(r"electricity|electric|storage heater|panel heater|electricaire",
                                                          regex=True,na=False)
epc_model.loc[electricity_category,"HEATING_SYSTEM_CLASS"] = "Electricity"

#classify heat pump heating systems
heat_pump_mask = combined_heating_text.str.contains(r"heat pump|ground source|air source|water source",
                                                    regex=True,na=False)
epc_model.loc[heat_pump_mask,"HEATING_SYSTEM_CLASS"] = "Heat pump"

#classify oil heating systems 
oil_category = combined_heating_text.str.contains(r"\boil\b",regex=True, na=False)
epc_model.loc[oil_category,"HEATING_SYSTEM_CLASS"] = "Oil"


#classify lpg heating systems
lpg_category = combined_heating_text.str.contains(r"\blpg\b|bottled gas|bulk gas", regex=True, na=False)
epc_model.loc[lpg_category,"HEATING_SYSTEM_CLASS"] = "LPG"

#classify biomass and solid fuel heating systems
biomass_mask = combined_heating_text.str.contains(r"wood|biomass|solid fuel|coal|anthracite|smokeless|biodiesel|pellet",
                                                  regex=True,na=False)
epc_model.loc[biomass_mask,"HEATING_SYSTEM_CLASS"] = "Biomass or solid fuel"

#classify mains gas heating systems
gas_category = combined_heating_text.str.contains(r"mains gas|natural gas|gas boiler",
                                            regex=True, na=False)
epc_model.loc[gas_category,"HEATING_SYSTEM_CLASS"] = "Mains gas"







In [ ]:
epc_model['HEATING_SYSTEM_CLASS'].unique()

In [ ]:
#summarize the heating system classes and their counts and percentages
summary = epc_model["HEATING_SYSTEM_CLASS"].value_counts().rename_axis("HEATING_SYSTEM_CLASS").reset_index(name="COUNT")
summary["Share_pct"] = (100 * summary["COUNT"] / summary["COUNT"].sum())
summary

In [ ]:
#check heatimg system with unknown classification
unknown = epc_model[epc_model["HEATING_SYSTEM_CLASS"] == "Unknown"]

unknown['MAINHEAT_DESCRIPTION'].value_counts()

In [ ]:
unknown['MAIN_FUEL'].value_counts()

In [ ]:
unknown['MAINS_GAS_FLAG'].value_counts()

In [ ]:
#remove community scheme records with unknown heating systems, main fuel
epc_model = epc_model[epc_model["HEATING_SYSTEM_CLASS"] != "Unknown"].copy()

In [ ]:
summary = epc_model["HEATING_SYSTEM_CLASS"].value_counts().rename_axis("HEATING_SYSTEM_CLASS").reset_index(name="COUNT")
summary["Share_pct"] = (100 * summary["COUNT"] / summary["COUNT"].sum())
summary

In [ ]:
#check main gas flag column for missing values
epc_model['MAINS_GAS_FLAG'].isnull().sum()

In [ ]:
epc_model['MAINS_GAS_FLAG'] = epc_model['MAINS_GAS_FLAG'].fillna('Missing')

In [ ]:
#resolve missing gas flags where the fuel clearly identifies mains gas

mains_gas_fuel = epc_model["MAIN_FUEL"].fillna("").astype(str).str.contains(r"mains gas", case=False, regex=True)

epc_model.loc[epc_model["MAINS_GAS_FLAG"].eq("Missing")& mains_gas_fuel,"MAINS_GAS_FLAG"] = "Y"

In [ ]:
#check the distribution of values in the MAINS_GAS_FLAG column
epc_model['MAINS_GAS_FLAG'].value_counts()

In [ ]:
F = epc_model[epc_model['MAINS_GAS_FLAG'] == 'Missing']
F[["MAINS_GAS_FLAG","MAIN_FUEL","MAINHEAT_DESCRIPTION"]].head(20)

In [ ]:
#compare heating systwm with main-gas availability 
gas_connection_table = pd.crosstab(
    epc_model["HEATING_SYSTEM_CLASS"],
    epc_model["MAINS_GAS_FLAG"],
    margins=True,
    dropna=False,
)
gas_connection_table

In [ ]:
# Identify individual non-gas heating system classes
individual_offgas_classes = ["Electricity","Heat pump","LPG","Oil","Biomass or solid fuel"]
epc_offgas = epc_model[epc_model["HEATING_SYSTEM_CLASS"].isin(individual_offgas_classes)].copy()
epc_offgas = epc_offgas[epc_offgas["MAINS_GAS_FLAG"] == "N"].copy()

print("Individual non-gas properties:", len(epc_offgas))

In [ ]:
epc_offgas[["HEATING_COST_CURRENT","HOT_WATER_COST_CURRENT"]].describe()

In [ ]:
#check fuels within each non-gas heating system class
offgas_fuel_counts = (
    epc_offgas
    .groupby("HEATING_SYSTEM_CLASS")["MAIN_FUEL"]
    .value_counts(dropna=False)
    .rename("COUNT")
    .reset_index()
)

offgas_fuel_counts

In [ ]:
E = epc_offgas[epc_offgas["HEATING_SYSTEM_CLASS"] == "Electricity"]

In [ ]:
C= E[['MAIN_FUEL','MAINHEAT_DESCRIPTION']]

E['MAINHEAT_DESCRIPTION'].str.contains(r"storage",regex=True,case=False,na=False).sum()

## 10. Load and prepare historical fuel prices


In [ ]:
fuel_price_history = pd.read_csv(FUEL_PRICE_HISTORY_PATH,parse_dates=["effective_from_date"],dayfirst=False,)
fuel_price_history.head()

In [ ]:
historical_prices = fuel_price_history.copy()

# Map historical fuel-price columns to your heating-system class names
column_map = {
    'effective_from_date': 'Date',
    "standard tariff": "Electricity",
    "mains gas": "Mains gas",
    "bulk LPG": "LPG",
    "heating oil": "Oil",
    "7-hour tariff low rate": "Electric storage heaters",
    "standard tariff standing charge": "Electricity_SC",
    "7-hour tariff low rate standing charge": "Electric storage heaters_SC",
    "mains gas standing charge": "Mains gas_SC",
    "bulk LPG standing charge": "LPG_SC",
    "bottled LPG standing charge": "bottled LPG_SC",
    "heating oil standing charge": "Oil_SC",
}

historical_prices = historical_prices.rename(columns=column_map)


# Heat pumps use the electricity tariff
historical_prices["Heat pump"] = historical_prices["Electricity"]
historical_prices["Heat pump_SC"] = historical_prices["Electricity_SC"]

# Representative price for the combined biomass/solid-fuel class
solid_fuel_columns = ["anthracite","dual fuel appliance","house coal","manufactured smokeless fuel","wood logs","wood pellets main heating"]
solid_charge_cols = ["anthracite standing charge","dual fuel appliance standing charge","house coal standing charge",
                     "manufactured smokeless fuel standing charge","wood logs standing charge", "wood pellets main heating standing charge",]
historical_prices["Biomass or solid fuel_SC"] = historical_prices[solid_charge_cols].median(axis=1)
historical_prices["Biomass or solid fuel"] = historical_prices[solid_fuel_columns].median(axis=1)




In [ ]:
fuel_price_columns = ["Electricity","Heat pump","Mains gas","LPG","Oil","Biomass or solid fuel","bottled LPG","Electric storage heaters",]

standing_charge_columns = ["Electricity_SC","Heat pump_SC","Mains gas_SC","LPG_SC","Oil_SC","Biomass or solid fuel_SC","bottled LPG_SC","Electric storage heaters_SC",]

historical_prices = historical_prices[["Date"] + fuel_price_columns + standing_charge_columns].copy()

# Convert only pence/kWh fuel prices to £/kWh
historical_prices[fuel_price_columns] = (historical_prices[fuel_price_columns] / 100)

In [ ]:
# Match prices to EPC lodgement dates
historical_prices = historical_prices.sort_values("Date")

epc_offgas = epc_offgas.sort_values("LODGEMENT_DATE")

epc_offgas = pd.merge_asof(epc_offgas,historical_prices,left_on="LODGEMENT_DATE",right_on="Date",direction="backward")

In [ ]:
fallback_columns = fuel_price_columns + standing_charge_columns

# Use the earliest available historical values for EPCs
# before the historical table begins
earliest_prices = historical_prices.iloc[0]

missing_price_date = epc_offgas["Date"].isna()

epc_offgas.loc[missing_price_date, "Date"] = earliest_prices["Date"]

for column in fallback_columns:
    epc_offgas.loc[missing_price_date, column] = earliest_prices[column]

print("EPCs using earliest available prices:", missing_price_date.sum())

In [ ]:
epc_offgas.head()

## 11. Space-heating section


In [ ]:
#assign fuel price to each property based on its heating system class
epc_offgas["FUEL_PRICE_GBP_KWH"] = epc_offgas.apply(
    lambda row: row[row["HEATING_SYSTEM_CLASS"]],
    axis=1
)

# Use bottled-LPG price for space heating only where the heating class is LPG

bottled_lpg = epc_offgas["MAIN_FUEL"].str.contains("bottled lpg",case=False,na=False)
space_bottled_lpg = (bottled_lpg& epc_offgas["HEATING_SYSTEM_CLASS"].eq("LPG"))
epc_offgas.loc[space_bottled_lpg,"FUEL_PRICE_GBP_KWH"] = epc_offgas.loc[space_bottled_lpg,"bottled LPG"]

#use 7-hour low tarrif price for storage heaters
storage_heaters = (epc_offgas["HEATING_SYSTEM_CLASS"].eq("Electricity") & epc_offgas["MAINHEAT_DESCRIPTION"].str.contains(
    "storage heaters",case=False, na=False))

epc_offgas.loc[storage_heaters,"FUEL_PRICE_GBP_KWH"] = epc_offgas.loc[storage_heaters,"Electric storage heaters"]


#standing charges are applied to electricity, storage heaters, mains gas, LPG, bottled LPG, and oil heating systems. Biomass and solid fuel systems do not have a standing charge.
epc_offgas["SPACE_STANDING_CHARGE_GBP"] = epc_offgas.apply(lambda row: row[f'{row["HEATING_SYSTEM_CLASS"]}_SC'],axis=1)

epc_offgas.loc[space_bottled_lpg, "SPACE_STANDING_CHARGE_GBP"] = epc_offgas.loc[space_bottled_lpg, "bottled LPG_SC"]

epc_offgas.loc[storage_heaters, "SPACE_STANDING_CHARGE_GBP"] = epc_offgas.loc[storage_heaters, "Electric storage heaters_SC"]


In [ ]:
#assign heating system efficiencies 
efficiency_values = {"Electricity": 1.00,"Heat pump": 2.78,"LPG": 0.84,
                     "Oil": 0.84,"Biomass or solid fuel": 0.82}

epc_offgas["HEATING_SYSTEM_EFFICIENCY"] = epc_offgas["HEATING_SYSTEM_CLASS"].map(efficiency_values)

## 12. Hot-water section


In [ ]:
#classify the hot-water fuel/system
#classify hot-water system from its description
hotwater_text = epc_offgas["HOTWATER_DESCRIPTION"].fillna("").astype(str).str.strip().str.lower()
epc_offgas["HOT_WATER_SYSTEM_CLASS"] = epc_offgas["HEATING_SYSTEM_CLASS"]

#electric hot-water systems
epc_offgas.loc[hotwater_text.str.contains(r"electric immersion|electric instantaneous|no system present|no hot water system present",
                                          regex=True,na=False),"HOT_WATER_SYSTEM_CLASS"] = "Electricity"

#heat pump must be assigned before general electric systems
epc_offgas.loc[hotwater_text.str.contains(r"heat pump", regex=True, na=False),"HOT_WATER_SYSTEM_CLASS"] = "Heat pump"

#gas hot-water systems
epc_offgas.loc[hotwater_text.str.contains(r"\bgas\b", regex=True, na=False),"HOT_WATER_SYSTEM_CLASS"] = "Mains gas"

#solid-fuel hot-water systems
epc_offgas.loc[
    hotwater_text.str.contains(r"solid fuel", regex=True, na=False),
    "HOT_WATER_SYSTEM_CLASS"
] = "Biomass or solid fuel"

#from main heating system
from_main = hotwater_text.str.startswith("from main system")

epc_offgas.loc[from_main,"HOT_WATER_SYSTEM_CLASS"] = epc_offgas.loc[from_main, "HEATING_SYSTEM_CLASS"]


In [ ]:
epc_offgas['HOT_WATER_SYSTEM_CLASS'].value_counts().rename_axis("HOT_WATER_SYSTEM_CLASS").reset_index(name="COUNT")

In [ ]:
#assign fuel price to each property based on its Hot water system class
epc_offgas["HOT_WATER_FUEL_PRICE_GBP_KWH"] = epc_offgas.apply(
    lambda row: row[row["HOT_WATER_SYSTEM_CLASS"]],
    axis=1
)
# Use bottled-LPG price for hot water only where the hot-water class is LPG
hotwater_bottled_lpg = (bottled_lpg & epc_offgas["HOT_WATER_SYSTEM_CLASS"].eq("LPG"))

epc_offgas.loc[hotwater_bottled_lpg,"HOT_WATER_FUEL_PRICE_GBP_KWH"] = epc_offgas.loc[hotwater_bottled_lpg,"bottled LPG"]


#use 7-hour low tarrif price for storage heaters
# Use the historical 7-hour low-rate tariff for off-peak electric hot water
offpeak_hotwater = (epc_offgas["HOT_WATER_SYSTEM_CLASS"].eq("Electricity") & epc_offgas["HOTWATER_DESCRIPTION"].str.contains("off-peak",case=False,na=False))

epc_offgas.loc[offpeak_hotwater,"HOT_WATER_FUEL_PRICE_GBP_KWH"] = epc_offgas.loc[offpeak_hotwater,"Electric storage heaters"]


epc_offgas["HOT_WATER_STANDING_CHARGE_GBP"] = epc_offgas.apply(
 lambda row: row[f'{row["HOT_WATER_SYSTEM_CLASS"]}_SC'],
    axis=1
)

epc_offgas.loc[hotwater_bottled_lpg, "HOT_WATER_STANDING_CHARGE_GBP"] = epc_offgas.loc[hotwater_bottled_lpg, "bottled LPG_SC"]

epc_offgas.loc[offpeak_hotwater, "HOT_WATER_STANDING_CHARGE_GBP"] = epc_offgas.loc[offpeak_hotwater, "Electric storage heaters_SC"]

In [ ]:
#assign provisional fuel prices and conversion factors
hot_water_efficiencies = {
    "Electricity": 1.00,
    "Heat pump": 2.78,
    "Mains gas": 0.84,
    "LPG": 0.84,
    "Oil": 0.84,
    "Biomass or solid fuel": 0.82
}


In [ ]:
# Standing-charge adjustment
#Avoid double-counting standing charges when the same fuel is used for both space heating and hot water. If the same fuel is used, we will combine the costs and subtract the higher of the two standing charges before allocating the remaining cost proportionally to space heating and hot water based on their respective shares of the total cost.
same_fuel = (
    (
        epc_offgas["HEATING_SYSTEM_CLASS"]
        == epc_offgas["HOT_WATER_SYSTEM_CLASS"]
    )
    & (space_bottled_lpg == hotwater_bottled_lpg)
    & (storage_heaters == offpeak_hotwater)
)

combined_cost = (
    epc_offgas["HEATING_COST_CURRENT"]
    + epc_offgas["HOT_WATER_COST_CURRENT"]
)

combined_adjusted = (
    combined_cost
    - epc_offgas[
        ["SPACE_STANDING_CHARGE_GBP", "HOT_WATER_STANDING_CHARGE_GBP"]
    ].max(axis=1)
).clip(lower=0)

space_share = epc_offgas["HEATING_COST_CURRENT"] / combined_cost
water_share = epc_offgas["HOT_WATER_COST_CURRENT"] / combined_cost

epc_offgas["ADJUSTED_HEATING_COST"] = np.where(
    same_fuel,
    combined_adjusted * space_share,
    (
        epc_offgas["HEATING_COST_CURRENT"]
        - epc_offgas["SPACE_STANDING_CHARGE_GBP"]
    ).clip(lower=0)
)

epc_offgas["ADJUSTED_HOT_WATER_COST"] = np.where(
    same_fuel,
    combined_adjusted * water_share,
    (
        epc_offgas["HOT_WATER_COST_CURRENT"]
        - epc_offgas["HOT_WATER_STANDING_CHARGE_GBP"]
    ).clip(lower=0)
)

## 13. Energy calculations


In [ ]:
#map hot-water efficiencies to the EPC off-gas dataset
epc_offgas["HOT_WATER_SYSTEM_EFFICIENCY"] = epc_offgas["HOT_WATER_SYSTEM_CLASS"].map(hot_water_efficiencies)

#calculate purchased energy from standing-charge-adjusted costs
epc_offgas["SPACE_HEAT_INPUT_KWH"] = epc_offgas["ADJUSTED_HEATING_COST"] / epc_offgas["FUEL_PRICE_GBP_KWH"]
epc_offgas["HOT_WATER_INPUT_KWH"] = epc_offgas["ADJUSTED_HOT_WATER_COST"] / epc_offgas["HOT_WATER_FUEL_PRICE_GBP_KWH"]

#apply efficiencies to convert purchased to useful heat
epc_offgas["SPACE_HEAT_USEFUL_KWH"] = epc_offgas["SPACE_HEAT_INPUT_KWH"] * epc_offgas["HEATING_SYSTEM_EFFICIENCY"]
epc_offgas["HOT_WATER_USEFUL_KWH"] = epc_offgas["HOT_WATER_INPUT_KWH"] * epc_offgas["HOT_WATER_SYSTEM_EFFICIENCY"]

#calculate useful space-heating intensity
epc_offgas["SPACE_HEAT_KWH_M2"] = epc_offgas["SPACE_HEAT_USEFUL_KWH"] / epc_offgas["TOTAL_FLOOR_AREA"]

#combine space heating and hot water
epc_offgas["TOTAL_USEFUL_HEAT_KWH"] = epc_offgas["SPACE_HEAT_USEFUL_KWH"] + epc_offgas["HOT_WATER_USEFUL_KWH"]

#calculate combined heat intensity
epc_offgas["TOTAL_USEFUL_HEAT_KWH_M2"] = epc_offgas["TOTAL_USEFUL_HEAT_KWH"] / epc_offgas["TOTAL_FLOOR_AREA"]

print(epc_offgas["HOT_WATER_SYSTEM_CLASS"].value_counts(dropna=False))

print("Missing combined heat estimates:", epc_offgas["TOTAL_USEFUL_HEAT_KWH"].isna().sum())

In [ ]:
#summarise combined useful heat demand by heating-system class
combined_heat_summary = (
    epc_offgas.groupby("HEATING_SYSTEM_CLASS")
    .agg(
        COUNT=("TOTAL_USEFUL_HEAT_KWH", "size"),
        MEAN_TOTAL_HEAT_KWH=("TOTAL_USEFUL_HEAT_KWH", "mean"),
        MEDIAN_TOTAL_HEAT_KWH=("TOTAL_USEFUL_HEAT_KWH", "median"),
        MIN_TOTAL_HEAT_KWH=("TOTAL_USEFUL_HEAT_KWH", "min"),
        MAX_TOTAL_HEAT_KWH=("TOTAL_USEFUL_HEAT_KWH", "max"),
        MEDIAN_TOTAL_HEAT_KWH_M2=("TOTAL_USEFUL_HEAT_KWH_M2", "median"),
        MAX_TOTAL_HEAT_KWH_M2=("TOTAL_USEFUL_HEAT_KWH_M2", "max")
    ).reset_index()
)

combined_heat_summary

In [ ]:
#check the EPC off-gas sample size in each LSOA
lsoa_epc_sample = (
    epc_offgas
    .groupby(["lsoa21cd", "lsoa21nm"])
    .size()
    .rename("OFFGAS_EPC_COUNT")
    .reset_index()
)

lsoa_epc_sample["OFFGAS_EPC_COUNT"].describe()

## 14. Aggregate EPC data to LSOA level


In [ ]:
# Aggregate off-grid EPC heat demand to all Plymouth LSOAs

all_lsoas = epc_model[["lsoa21cd", "lsoa21nm"]].drop_duplicates()

local_epc_summary = (epc_offgas.groupby(["lsoa21cd", "lsoa21nm"], as_index=False).agg(OFFGAS_EPC_COUNT=("PROPERTY_ID", "size"),
                                                                                      MEDIAN_HEAT_PER_PROPERTY_KWH=("TOTAL_USEFUL_HEAT_KWH","median" )))

lsoa_epc_summary = all_lsoas.merge(local_epc_summary,on=["lsoa21cd", "lsoa21nm"],how="left",validate="one_to_one")

lsoa_epc_summary["OFFGAS_EPC_COUNT"] = (lsoa_epc_summary["OFFGAS_EPC_COUNT"].fillna(0).astype(int))
lsoa_epc_summary.describe()

In [ ]:
lsoa_epc_summary.sort_values("OFFGAS_EPC_COUNT", ascending=True).head(20)

In [ ]:
#use Plymouth-wide median where the LSOA EPC sample is below 10
plymouth_median_heat = epc_offgas["TOTAL_USEFUL_HEAT_KWH"].median()

lsoa_epc_summary["ADJUSTED_MEDIAN_HEAT_PER_PROPERTY_KWH"] = np.where(
    lsoa_epc_summary["OFFGAS_EPC_COUNT"] >= 10,
    lsoa_epc_summary["MEDIAN_HEAT_PER_PROPERTY_KWH"],
    plymouth_median_heat
)

lsoa_epc_summary["HEAT_ESTIMATE_SOURCE"] = np.where(lsoa_epc_summary["OFFGAS_EPC_COUNT"] >= 10,"LSOA median","Plymouth median")

print("Plymouth median heat:", plymouth_median_heat)

lsoa_epc_summary[
    "HEAT_ESTIMATE_SOURCE"
].value_counts()

In [ ]:
#retain final EPC demand estimate for each LSOA
required_columns =["lsoa21cd","lsoa21nm", "OFFGAS_EPC_COUNT", "MEDIAN_HEAT_PER_PROPERTY_KWH", "ADJUSTED_MEDIAN_HEAT_PER_PROPERTY_KWH","HEAT_ESTIMATE_SOURCE"]
lsoa_epc_output = lsoa_epc_summary[required_columns].copy()

lsoa_epc_output.head()

In [ ]:
energy_epc_lsoa_path = PROCESSED_EPC_DIR / "plymouth_offgas_energy_epc_lsoa_2024.csv"

lsoa_epc_output.to_csv(energy_epc_lsoa_path, index=False)
